# Money Leak Backtest — SEITH (research isolated)
Plotly offline fig.show() — 7 cells, no apps/kronos-sidecar pollution (grep plotly →0).
Data: research/universe-100.json (100 stratified) → Sectors /v2/daily/{symbol}/ (fallback tests/fixtures/bbca-ohlcv-400.json) → normalize (OHLC→excluded/volume→0/median) → Kronos 400→20 (fallback kronos-pred-20.json) → Score 30/20/30/20 → Flag |Z|>2 → Rank → research/backtest-100.json

In [ ]:
import json, pathlib
uni = json.loads(pathlib.Path('research/universe-100.json').read_text())
assert len(uni['items']) == 100, len(uni['items'])
assert uni['stratified'] == {'FINANCE':25,'ENERGY':20,'CONSUMER':20,'INFRA':20,'OTHER':15}
print(f"universe {uni['as_of']} {len(uni['items'])} OK")

In [ ]:
# 02 normalize + sector-median fallback per market (Id vs Sg terpisah)
import json, pathlib
med = json.loads(pathlib.Path('tests/fixtures/sector-median.json').read_text())
ill = json.loads(pathlib.Path('tests/fixtures/illiquid-ohlcv.json').read_text())
print('sector-median', list(med.get('id',{}).keys())[:3], 'illiquid', len(ill))
# OHLC 0/nan → excluded, volume None→0, rasio None→median, lookback>512→422 (see normalize.rs)

In [ ]:
# 03 Kronos 400→20 zero-shot (KRONOS_MOCK=1 for CI, real predict_batch T1.0 top_p0.9 max_context 512)
import json, pathlib
pred = json.loads(pathlib.Path('tests/fixtures/kronos-pred-20.json').read_text())
print('kronos-pred-20', len(pred), pred[0] if pred else 'empty')
# live: POST :8001/predict_batch {market, df, x_timestamp, y_timestamp, pred_len:20, T:1.0, top_p:0.9}

In [ ]:
# 04-05 Score 0-100 (30ER+20(100-|Z|)+30QV+20SM clamp) + Flag |Z|>2 OR vol>2σ → Rank
import json, pathlib
bt = json.loads(pathlib.Path('research/backtest-100.json').read_text())
assert len(bt['items']) == 100
top5 = sorted([x for x in bt['items'] if x['anomaly']['flag']], key=lambda x: abs(x['anomaly']['z']), reverse=True)[:5]
print('backtest', bt['as_of'], 'items', len(bt['items']), 'top5 flags', len(top5), 'metrics', bt['metrics'])
# rank Mispricing desc → |Z| tie-break; anomaly sort |Z| desc → score tie-break (see ranking/service.rs)

In [ ]:
# 06 metrics (Sharpe/maxDD/win rate) — verifiable
import json
m = json.loads(open('research/backtest-100.json').read())['metrics']
print({k: round(float(v),4) for k,v in m.items() if isinstance(v,(int,float))})
assert 0 <= m['hit_rate'] <= 1
assert m['drawdown'] <= 0

In [ ]:
# 07 equity vs IHSG plot (plotly 5.* isolated, offline)
try:
    import plotly.graph_objects as go
    import json
    bt = json.loads(open('research/backtest-100.json').read())
    ec = bt.get('equity_curve', [])
    fig = go.Figure()
    if ec:
        fig.add_trace(go.Scatter(x=[r['date'] for r in ec], y=[r['return'] for r in ec], name='SEITH'))
        fig.add_trace(go.Scatter(x=[r['date'] for r in ec], y=[r['bench'] for r in ec], name='IHSG'))
    fig.update_layout(title='Equity vs IHSG (research isolated)', template='plotly_dark')
    fig.show()
    print('plotly', go.__version__, 'traces', len(fig.data))
except Exception as e:
    print('plotly fallback (no display):', e)
    print('equity_curve len', len(json.loads(open('research/backtest-100.json').read()).get('equity_curve',[])))